### From Tif to DataFrame:

This notebook has all the work that goes into taking our tif results from the py file to a usable GeoDataFrame for training our models.

Below you will find:
1. [Processing the data and upscaling](#processing-the-data-and-upscaling)
    - First getting all the initial needed columns
    - [secondary column additions](#secondary-column-additions)
2. [Merging the DataFrames with the target data](#merging-the-dataframes-with-the-target-data)
3. [Saving the results](#clean-and-save-data-for-training)

**import needed libraries:**

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import geopandas as gpd
matplotlib.use('Agg')
import numpy as np
import rasterio
import sys
import os, tempfile
from osgeo import ogr  
ogr.UseExceptions()
from datetime import datetime, timedelta
import warnings
import shapefile
import rioxarray
import pandas as pd
import csv
from rasterio.crs import CRS
from rasterio.merge import merge
from shapely.geometry import shape
from shapely import wkt
import geopandas as gpd
from shapely.geometry import Point
from shapely.strtree import STRtree
from shapely.geometry import box
import psutil
import shutil
from pathlib import Path
import re
from sklearn.model_selection import train_test_split
import json
import xarray as xr
%load_ext autoreload
%autoreload 2



from helper_functions import *

#### Define needed functions for the work in the cell below. 


Starting with functions to get the different tif files into the correct dimension:

In [ ]:
def upscale_data(gdf, input_raster, method=["mean"],duration=None):
    '''
    this function takes in a grid geodataframe and a raster file, and 
    calculates the specified zonal statistic for each grid cell, then saves the results 
    as a new column in the geodataframe. The method can be "mean", "median", "std", etc. 
    If method is "std", it calculates the standard deviation by first calculating the mean of the 
    squared values and then taking the square root.
    '''
    # Open raster, mask out both nodata values
    with rasterio.open(input_raster) as src:
        data = src.read(1).astype(float)
        data[data == -9999] = np.nan
        data[data == 888] = np.nan
        data[data == 999] = np.nan
        data[data == 9999] = np.nan
        # write to temp file
        profile = src.profile.copy()
        profile.update({"dtype": "float64", "nodata": np.nan})
        temp_file = "../data/temp_masked.tif"
        with rasterio.open(temp_file, "w", **profile) as dst:
            dst.write(data, 1)

    if method == 'categorical':
        stats = zonal_stats(gdf, temp_file, stats=[], categorical=True, nodata=np.nan)
        # gdf[column_name] = stats
    elif method == 'percentage_avail':
        sum_stats = zonal_stats(gdf, temp_file, stats=["sum", "count"], nodata=np.nan)
        stats = [(s["sum"] / (s["count"] * duration)) if s["count"] and s["count"] > 0 else None for s in sum_stats]
    else:
        stats_org = zonal_stats(gdf, temp_file, stats=method, nodata=np.nan)
        # save the results as a column to the geodataframe
        stats = [s[method] if s[method] is not None else None for s in stats_org]
    
    return stats

def flag_proportion(gdf, input_raster, flag_value):
    '''
    This function takes a geodatafram, an input raster, and a flag value. It determines the proportion
    of the pixels in the cell that are equal to the flag value, and returns that percentage
    in a decimal format (between 0 and 1). 
    '''
    with rasterio.open(input_raster) as src:
        data = src.read(1).astype(float)

        if flag_value == -9999:
            flag_raster = np.where((data == flag_value) | (np.isnan(data)), 1.0, 0.0)
        else:
            flag_raster = np.where(data == flag_value, 1.0, 0.0)
        # flag_raster[data == NODATA_VALUE] = np.nan #getting rid of this
        profile = src.profile.copy()
        profile.update({"dtype": "float64"})
        temp_file = "../data/temp_flag.tif"
        with rasterio.open(temp_file, "w", **profile) as dst:
            dst.write(flag_raster, 1)
    stats = zonal_stats(gdf, temp_file, stats=["median"], nodata=np.nan)
    return [s["median"] for s in stats]

def affected_perc_and_bool(aff_col):
    '''
    this function takes in the column name for the affected percentage, 
    and creates two new columns: one for the percentage and one for a boolean 
    indicating whether the cell was affected or not. It assumes that the categories 
    column is in the format "{0: 0.0, 1: 0.5, 2: 0.5}" where the keys are the category 
    values and the values are the number of cells in that category. The percetage will 
    be calulated as (number of affected cells / total number of cells) (in decimal).
    '''
    
    if aff_col is None or aff_col == '' or aff_col == '{}':
        return 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
    cat_dict = {i.split(':')[0].strip(): i.split(':')[1].strip() for i in aff_col.replace('{','').replace('}','').split(',')}  
    total_cells = sum(float(v) for v in cat_dict.values())
    affected_cells = sum(float(v) for k, v in cat_dict.items() if k in ['1.0', '2.0'])
    possibly_aff_cells = sum(float(v) for k, v in cat_dict.items() if k in ['2.0'])
    cert_aff_cells = sum(float(v) for k, v in cat_dict.items() if k in ['1.0'])
    not_affected_cells = sum(float(v) for k, v in cat_dict.items() if k in ['3.0'])
    four_affected_cells = sum(float(v) for k, v in cat_dict.items() if k in ['4.0'])
    zero_affected_cells = sum(float(v) for k, v in cat_dict.items() if k in ['0.0'])
    affected_perc = (affected_cells / total_cells) if total_cells > 0 else 0
    cert_not_affected_perc = (not_affected_cells / total_cells) if total_cells > 0 else 0
    poss_affected_perc = (possibly_aff_cells / total_cells) if total_cells > 0 else 0
    cert_aff_perc = (cert_aff_cells / total_cells) if total_cells > 0 else 0
    four_aff_perc = (four_affected_cells / total_cells) if total_cells > 0 else 0
    zero_aff_perc = (zero_affected_cells / total_cells) if total_cells > 0 else 0
    cert_aff = 1 if cert_aff_cells > 0 else 0 #want these to be 1/0 for the model, not True/False
    prob_aff = 1 if affected_cells > 0 else 0
    cert_not_aff = 1 if not_affected_cells > 0 else 0

    #make this a list so it's appened for pep reasons
    return 0, affected_perc, cert_aff, prob_aff, cert_not_aff, cert_not_affected_perc, poss_affected_perc, cert_aff_perc, four_aff_perc, zero_aff_perc, total_cells
        
def is_leap_year(year):
    """ Check if a year is a leap year """
    return (year % 400 == 0) or ((year % 4 == 0) and (year % 100 != 0))

import tempfile

def write_temp_tif(data_2d, haz_tif, nodata=np.nan):
    # accept either a path string or an open rasterio dataset
    if isinstance(haz_tif, str):
        with rasterio.open(haz_tif) as src:
            profile = src.profile.copy()
    else:
        profile = haz_tif.profile.copy()
    
    profile.update({'dtype': 'float64', 'nodata': nodata, 'count': 1})
    tmp = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
    with rasterio.open(tmp.name, 'w', **profile) as dst:
        dst.write(data_2d, 1)
    return tmp.name

def mask_tif_with_urban(input_tif_path, urban_mask_array):
    with rasterio.open(input_tif_path) as src:
        data    = src.read(1).astype(float)
        profile = src.profile.copy()
    data[urban_mask_array != 1] = np.nan
    data[np.isnan(urban_mask_array)] = np.nan
    profile.update({'dtype': 'float64', 'nodata': np.nan})
    tmp = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
    with rasterio.open(tmp.name, 'w', **profile) as dst:
        dst.write(data, 1)
    return tmp.name

def _open_nc_tile(base_path,year,tile):
    key = (int(year), tile)
    if key in nc_cache:
        return nc_cache[key]
    nc_path = f"{base_path}/{year}/{tile}_{year}.nc"
    # print(f"DEBUG: Looking for NC file at: {nc_path}")
    # print(f"DEBUG: File exists: {os.path.exists(nc_path)}")
    if not os.path.exists(nc_path):
        nc_cache[key] = None
        return None
    ds = xr.open_dataset(nc_path)
    nc_cache[key] = ds
    return ds

**Defining variables used across this notebook:**

In [13]:
storm_names_simp = [
    'bopha',
    'utor',
    'trami',
    'usagi',
    'nari',
    'krosa',
    'haiyan',
    'lingling',
    'sarika',
    'haima',
    'nock-ten',
    'mangkhut'
] # the ones I know we have data for

storm_names = [
    'bopha_0',
    'utor_1',
    'trami_2',
    'usagi_3',
    'nari_4',
    'krosa_5',
    'haiyan_6',
    'lingling_7',
    'sarika_18',
    'haima_19',
    'nock_ten_20',
    'mangkhut_21'
]

region = "philippines"
output_base = f"data/output_urban/"
post_base = f"{output_base}results/postprocessing/"
data_base = f"{output_base}"
base_path = "../data/tiles" 

hazard_tif = f"../data/vectors/philippines.tif" #need to get a file for the philipines
BUFFER_pre = 150 
BUFFER_post = 150
NODATA_VALUE = -9999
RECOVERY_NOT_FOUND = 999
NOT_ENOUGH_DATA = 888

events_df = pd.read_csv(f"../data/typhoon_dates.csv") #.iloc[20:21,:] #just the first event for testing
events = events_df[events_df.STORMNAME.str.lower().isin(storm_names_simp)] #filtering for these events

**Getting the correct clipped grid for our results for our model:**

This only needs to be run _once_ and then can be read in for followed up work or if this work needs to be done again. This process takes some time to complete.

In [ ]:
grid = gpd.read_file("../data/shape_files/phl_0.1_degree_grid.gpkg")
philippines = gpd.read_file("../data/shape_files/buffered_files/PH_dissolved_buffer.shp")

#make sure CRS matches
philippines = philippines.set_crs("EPSG:32651")
philippines = philippines.to_crs(grid.crs)

#clip grid to philippines boundary (don't want extra cells)
grid_clipped = grid.clip(philippines)

# Save the GeoDataFrame to a GeoPackage
grid_clipped.to_file('../data/shape_files/clipped_grid_for_analysis.gpkg', driver='GPKG')

# checking to make sure the shape is fine
#grid_clipped.shape, grid.shape
#output: ((3798, 5), (21042, 5)) -- looks good; also checked visually in QGIS

**Getting durations and analysis times:**

These are only important for some of the avaiblility metrics that are computed. Technically this can be done in the original py file, however my machine doesn't have enough memory to run all storms in one go, sadly.

In [176]:
# dictionary of duration_times for the event
duration_times = {}
analysis_times = {}

for event in events.itertuples():
    print(event)

    # print(f"Processing event: {event.STORMNAME}")

    region = f"{region}_{event.STORMNAME.lower().replace('-', '_')}_{event.Index}"
    sdate = pd.to_datetime(event.start_time)
    edate = pd.to_datetime(event.end_time)

        
    #Extent analysis window to include potential signals from evacuations or delayed impacts
    sdate = sdate - pd.Timedelta(days=3) #Start 3 days before event start
    edate = edate + pd.Timedelta(days=5) #End 5 days after event end

    syear, smonth, sday = sdate.year, sdate.month, sdate.day
    eyear, emonth, eday = edate.year, edate.month, edate.day

    print(f"Processing event: {eyear},{syear}")

    #date calculation
    months=[0,31,59,90,120,151,181,212,243,273,304,334] #Months expressed in DOY

    #Caculate start and end dates in DOY
    sd_doy = months[smonth-1] + sday #start from 1
    ed_doy = months[emonth-1] + eday

    #Ensure accuracy for leap years
    idx = 0
    if is_leap_year(syear):
        idx = 1 #leap year
    if (idx==1) and (smonth > 2):
        sd_doy += 1 
    idx = 0
    days_in_yr = 365
    if is_leap_year(eyear):
        idx = 1
        days_in_yr = 366
    if (idx == 1) and (emonth > 2):
        ed_doy += 1

    
    duration_event = (edate - sdate).days + 1 #general statement that works under any circumstance
    total_days_analysis = duration_event + BUFFER_pre + BUFFER_post 

    print(duration_event)
    duration_times[event.STORMNAME] = duration_event
    analysis_times[event.STORMNAME] = total_days_analysis

with open("../data/dicts_from_analysis/duration_times_dict.json", "w") as f:
        json.dump(duration_times, f, indent=2)

with open("../data/dicts_from_analysis/analysis_times_dict.json", "w") as f:
        json.dump(analysis_times, f, indent=2)

Pandas(Index=0, STORMNAME='BOPHA', start_time='2012-11-25 18:00:00', end_time='2012-12-09 6:00:00')
Processing event: 2012,2012
22
Pandas(Index=1, STORMNAME='UTOR', start_time='2013-08-08 12:00:00', end_time='2013-08-18 6:00:00')
Processing event: 2013,2013
18
Pandas(Index=2, STORMNAME='TRAMI', start_time='2013-08-16 12:00:00', end_time='2013-08-24 6:00:00')
Processing event: 2013,2013
16
Pandas(Index=3, STORMNAME='USAGI', start_time='2013-09-16 0:00:00', end_time='2013-09-24 0:00:00')
Processing event: 2013,2013
17
Pandas(Index=4, STORMNAME='NARI', start_time='2013-10-08 12:00:00', end_time='2013-10-16 12:00:00')
Processing event: 2013,2013
17
Pandas(Index=5, STORMNAME='KROSA', start_time='2013-10-27 18:00:00', end_time='2013-11-04 18:00:00')
Processing event: 2013,2013
17
Pandas(Index=6, STORMNAME='HAIYAN', start_time='2013-11-03 6:00:00', end_time='2013-11-11 6:00:00')
Processing event: 2013,2013
17
Pandas(Index=7, STORMNAME='LINGLING', start_time='2014-01-15 0:00:00', end_time='201

### Processing the data and upscaling

The subsequent cell does the marjority of the upscaling of the data. 

There are following cells that came later to try to add more features to our data.

In [178]:
duration_times = pd.read_json("../data/dicts_from_analysis/duration_times_dict.json", typ='series')

recovery_vars = [
    '_recov_dur_avail50',
    '_recov_dur_pers_avail50',
    '_recov_dur_avail30',
    '_recov_dur_pers_avail30',
    '_recov_day_pers',
    '_recov_day',
    '_recov_dur',
    '_recov_dur_pers'
]

for name in storm_names:
    print(f"Processing {name}...")
    files = [
        f'../{output_base}affected/philippines_{name}_aff.tif',
        f'../{output_base}available_percentage/philippines_{name}_available.tif',
        f'../{output_base}DPmax_150/philippines_{name}_DPmax.tif',
        f'../{output_base}pre_mean/philippines_{name}_pre_event_mean.tif',
        f'../{output_base}results/postprocessing/impact_duration/philippines_{name}_impact_dur_avail50.tif',
        f'../{output_base}results/postprocessing/impact_duration/philippines_{name}_impact_dur_pers_avail50.tif',
        f'../{output_base}results/postprocessing/impact_duration/philippines_{name}_impact_dur_avail30.tif',
        f'../{output_base}results/postprocessing/impact_duration/philippines_{name}_impact_dur_pers_avail30.tif',
        # f'../data/output_new/results/postprocessing/postimpact_availability/philippines_{name}_postimpact_avail.tif', # don't need, maybe think about it
        f'../{output_base}results/postprocessing/preevent_availability/philippines_{name}_preevent_avail.tif',
        f'../{output_base}results/postprocessing/recovery_duration/philippines_{name}_recov_dur_avail50.tif',
        f'../{output_base}results/postprocessing/recovery_duration/philippines_{name}_recov_dur_avail30.tif',
        f'../{output_base}results/postprocessing/recovery_duration/philippines_{name}_recov_dur_pers_avail50.tif',
        f'../{output_base}results/postprocessing/recovery_duration/philippines_{name}_recov_dur_pers_avail30.tif',
        f'../{output_base}sd_NTL/philippines_{name}_sd.tif',
        f'../{output_base}available_cell_counts/philippines_{name}_available_cell_counts.tif'
    ]

    #use also the 30% files and not the 50% file
    gdf = gpd.read_file('../data/shape_files/clipped_grid_for_analysis.gpkg')
    for f in files:
        var_name = f.split(name)[-1].replace('.tif','')
        # print(f.split(name)[-1])

        if var_name in recovery_vars:
            # flag columns first
            if var_name in ['_recov_day_pers', '_recov_day']:
                gdf[f"prop_9999{var_name}"] = flag_proportion(gdf, f, 9999)

            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, f, NODATA_VALUE)
            #recover not found has been removed this can be removed
            # gdf[f"prop_recovery_not_found{var_name}"] = flag_proportion(gdf, f, RECOVERY_NOT_FOUND)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, f, NOT_ENOUGH_DATA)
            # then the actual metric with special values masked
            for m in ["mean","median"]: #lets do means here
                column_name = f"{m}{var_name}"
                gdf[column_name] = upscale_data(gdf, f, m)
        elif var_name == '_aff':
            method = 'categorical'
            column_name = f"categories{var_name}"
            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, f, NODATA_VALUE)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, f, NOT_ENOUGH_DATA)
            gdf[column_name] = upscale_data(gdf, f, method)

            column_name = f"majority{var_name}"
            gdf[column_name] = upscale_data(gdf, f, 'majority')

            column_name = f"minority{var_name}"
            gdf[column_name] = upscale_data(gdf, f, 'minority')

        elif var_name == '_available_cell_counts':
            if name == 'nock_ten_20':
                n = 'NOCK-TEN'
                dur = int(duration_times[n])
            else:
                n = name.split('_')[0].upper()
                dur = int(duration_times[n])

            column_name = "perc_available_upscaled_new"
            gdf[column_name] = upscale_data(gdf, f, 'percentage_avail', dur)

        else: 
            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, f, NODATA_VALUE)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, f, NOT_ENOUGH_DATA)
            for m in ["mean", "median"]:
                column_name = f"{m}{var_name}"
                gdf[column_name] = upscale_data(gdf, f, m)

    # after all the upscale_data calls, before saving
    ntl_cols = [c for c in gdf.columns if c not in ['id', 'Longitude', 'Latitude', 'Centroid', 'geometry']]  # or whatever your non-data cols are
    gdf['all_ntl_null'] = gdf[ntl_cols].isnull().all(axis=1)
    os.makedirs(f'../data/upscale_data_urban/{name.split("_")[0]}', exist_ok=True)
    gdf.to_file(f'../data/upscale_data_urban/{name.split("_")[0]}/grided_data_for_{name}.gpkg', driver='GPKG')

Processing bopha_0...
Processing utor_1...
Processing trami_2...
Processing usagi_3...
Processing nari_4...
Processing krosa_5...
Processing haiyan_6...
Processing lingling_7...
Processing sarika_18...
Processing haima_19...
Processing nock_ten_20...
Processing mangkhut_21...


In [170]:
gdf = gpd.read_file('../data/shape_files/clipped_grid_for_analysis.gpkg')

In [172]:
gdf.columns

Index(['id', 'Longitude', 'Latitude', 'Centroid', 'geometry'], dtype='str')

Same process, but masked for only urban and suburban areas:

In [149]:
# load once outside storm loop
nc_urban = xr.open_dataset(f"../{output_base}/landcover/{region}_smod_urban_interp.nc")

for name in storm_names:
    print(f"Processing {name}...")
    if name == 'nock_ten_20':
        storm_name = 'NOCK-TEN'
    else:
        storm_name = name.split("_")[0].upper()

    storm_year   = int(events[events['STORMNAME'] == storm_name].start_time.iloc[0][:4])
    clamped_year = max(2010, min(2020, storm_year))
    urban_slice  = nc_urban["smod_urban"].sel(year=clamped_year).values

    files = [
        f'../data/output_new/affected/philippines_{name}_aff.tif',
        f'../data/output_new/available_percentage/philippines_{name}_available.tif',
        f'../data/output_new/DPmax_150/philippines_{name}_DPmax.tif',
        f'../data/output_new/pre_mean/philippines_{name}_pre_event_mean.tif',
        f'../data/output_new/results/postprocessing/impact_duration/philippines_{name}_impact_dur_avail50.tif',
        f'../data/output_new/results/postprocessing/impact_duration/philippines_{name}_impact_dur_pers_avail50.tif',
        f'../data/output_new/results/postprocessing/impact_duration/philippines_{name}_impact_dur_avail30.tif',
        f'../data/output_new/results/postprocessing/impact_duration/philippines_{name}_impact_dur_pers_avail30.tif',
        f'../data/output_new/results/postprocessing/preevent_availability/philippines_{name}_preevent_avail.tif',
        f'../data/output_new/results/postprocessing/recovery_duration/philippines_{name}_recov_dur_avail50.tif',
        f'../data/output_new/results/postprocessing/recovery_duration/philippines_{name}_recov_dur_avail30.tif',
        f'../data/output_new/results/postprocessing/recovery_duration/philippines_{name}_recov_dur_pers_avail50.tif',
        f'../data/output_new/results/postprocessing/recovery_duration/philippines_{name}_recov_dur_pers_avail30.tif',
        f'../data/output_new/sd_NTL/philippines_{name}_sd.tif',
        f'../data/output_new/available_cell_counts/philippines_{name}_available_cell_counts.tif'
    ]

    gdf = gpd.read_file('../data/shape_files/clipped_grid_for_analysis.gpkg')

    for f in files:
        var_name = f.split(name)[-1].replace('.tif', '')
        masked_f = mask_tif_with_urban(f, urban_slice)

        if var_name in recovery_vars:
            if var_name in ['_recov_day_pers', '_recov_day']:
                gdf[f"prop_9999{var_name}"] = flag_proportion(gdf, masked_f, 9999)
            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, masked_f, NODATA_VALUE)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, masked_f, NOT_ENOUGH_DATA)
            for m in ["mean", "median"]:
                gdf[f"{m}{var_name}"] = upscale_data(gdf, masked_f, m)

        elif var_name == '_aff':
            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, masked_f, NODATA_VALUE)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, masked_f, NOT_ENOUGH_DATA)
            gdf[f"categories{var_name}"] = upscale_data(gdf, masked_f, 'categorical')
            gdf[f"majority{var_name}"]   = upscale_data(gdf, masked_f, 'majority')
            gdf[f"minority{var_name}"]   = upscale_data(gdf, masked_f, 'minority')

        elif var_name == '_available_cell_counts':
            if name == 'nock_ten_20':
                n   = 'NOCK-TEN'
                dur = int(duration_times[n])
            else:
                n   = name.split('_')[0].upper()
                dur = int(duration_times[n])
            gdf["perc_available_upscaled_new"] = upscale_data(gdf, masked_f, 'percentage_avail', dur)

        else:
            gdf[f"prop_NODATA_VALUE{var_name}"] = flag_proportion(gdf, masked_f, NODATA_VALUE)
            gdf[f"prop_not_enough_data{var_name}"] = flag_proportion(gdf, masked_f, NOT_ENOUGH_DATA)
            for m in ["mean", "median"]:
                gdf[f"{m}{var_name}"] = upscale_data(gdf, masked_f, m)

        os.unlink(masked_f)

    os.makedirs(f'../data/upscale_data_final_urban/{name.split("_")[0]}', exist_ok=True)
    gdf.to_file(
        f'../data/upscale_data_final_urban/{name.split("_")[0]}/grided_data_for_{name}_urban.gpkg',
        driver='GPKG'
    )
    print(f"Saved urban-masked file for {name}")

nc_urban.close()

Processing bopha_0...
Saved urban-masked file for bopha_0
Processing utor_1...
Saved urban-masked file for utor_1
Processing trami_2...
Saved urban-masked file for trami_2
Processing usagi_3...
Saved urban-masked file for usagi_3
Processing nari_4...
Saved urban-masked file for nari_4
Processing krosa_5...
Saved urban-masked file for krosa_5
Processing haiyan_6...
Saved urban-masked file for haiyan_6
Processing lingling_7...
Saved urban-masked file for lingling_7
Processing sarika_18...
Saved urban-masked file for sarika_18
Processing haima_19...
Saved urban-masked file for haima_19
Processing nock_ten_20...
Saved urban-masked file for nock_ten_20
Processing mangkhut_21...
Saved urban-masked file for mangkhut_21


### EDA: Exploratory Data Analysis & Feature Engineering

Next we'll be doing some feature engineering based on the datasets we have from the last step. First, some data exploration, looking over what everything is currently looking like.

This wasn't much but I explored the data some to try to think through some needed columns, the following cell can be ignored mostly.

Creating new columns that will be useful in the final analysis and dropping the ones that need to be removed for our final csv.

In [ ]:
bopha_gdf = gpd.read_file('../data/upscale_data_final/bopha/grided_data_for_bopha_0.gpkg')

cat_dict = {i.split(':')[0].strip(): i.split(':')[1].strip() for i in bopha_gdf['categories_aff'].iloc[0].replace('{','').replace('}','').split(',')}  
total_cells = sum(float(v) for v in cat_dict.values())
affected_cells = sum(float(v) for k, v in cat_dict.items() if k in ['1.0', '2.0'])
affected_perc = (affected_cells / total_cells) if total_cells > 0 else 0
for_sure_aff = sum(float(v) for k, v in cat_dict.items() if k in ['1.0']) > 0 
prob_aff = affected_cells > 0

#saving the column names to a csv for reference - there were a lot of columns so wanted to view easier
pd.Series([i for i in bopha_gdf.columns]).to_csv('../data/upscale_data_final/bopha/bopha_columns.csv', index=False)

# [events['STORMNAME'] == "UTOR"].start_time.iloc[0][:4]

# events.STORMNAME.unique

### Secondary column additions

This following cell adds additional columns that hopefully give more information about how the grids compare when it comes to the affected cell columns. These include:
- `affected_perc`: percep 
- `cert_aff`: cell contains certainly affected pixels
- `prob_aff`: cell contains probably affected pixels
- `cert_not_aff`: cell contains certainly not affected pixels
- `cert_not_aff_perc`: percent of pixels in cell that were certainly not affected - Cat 3
- `pot_affected_perc`: percent of pixels in cell that were potentially affected - 
- `cert_aff_perc`: percent of pixels in cell that were certainly affected - Cat 1
- `aff_cat_four_perc`: percent of pixels in cell that were not affected - Cat 4
- `aff_cat_zero_perc`: percent of pixels in cell that were not affected - Cat 0
- `total_cells_for_aff`: total number of pixels in the cell

Categories:
- `1` - Certainly impacted cells
- `2` - Potentially impacted because low data availability, but had an impacted neighbor
- `3` - Certainly not impacted cell, had data availability and still shown as not impacted
- `4` - Uncertain: low data availability with no affected neighbors
- `0` - Uncertain: does not fall into any of the other categories.

In [179]:
# for clarity, storm name list with number included like: `bopha_0`
path = '../data/upscale_data_urban/'
for storm in storm_names:
    print(storm)

    if storm == 'nock_ten_20':
        storm_name = 'NOCK-TEN'
    else:
        storm_name = storm.split("_")[0].upper()
    file = f'{path}{storm.split("_")[0]}/grided_data_for_{storm}.gpkg'
    gdf_update = gpd.read_file(file)
    gdf_update['typhoon_name'] = storm_name
    gdf_update['typhoon_year'] = events[events['STORMNAME'] == storm_name].start_time.iloc[0][:4]
    gdf_update['grid_point_id'] = gdf_update['id'] #not sure why I didn't do a normal rename function ??????
    aff_list = ['no_aff_data','affected_perc', 'cert_aff', 'prob_aff', 'cert_not_aff',
                'cert_not_aff_perc','pot_affected_perc','cert_aff_perc',
                'aff_cat_four_perc','aff_cat_zero_perc', 'total_cells_for_aff']
    gdf_update[aff_list] = gdf_update.apply(lambda row: affected_perc_and_bool(row['categories_aff']), axis=1, result_type='expand')
    gdf_update.to_file(f'{path}{storm.split("_")[0]}/grided_data_for_{storm}_expanded.gpkg', driver='GPKG')
    gdf_update.drop(columns=['geometry','Longitude','Latitude','Centroid','categories_aff','id']).to_csv(f'{path}{storm.split("_")[0]}/grided_data_for_{storm}_expanded.csv', index=False)

bopha_0
utor_1
trami_2
usagi_3
nari_4
krosa_5
haiyan_6
lingling_7
sarika_18
haima_19
nock_ten_20
mangkhut_21


In [156]:
pd.read_csv('../data/upscale_data_final_urban/bopha/grided_data_for_bopha_0_expanded_urban_2.csv')

,prop_NODATA_VALUE_aff,prop_not_enough_data_aff,majority_aff,minority_aff,prop_NODATA_VALUE_available,prop_not_enough_data_available,mean_available,median_available,prop_NODATA_VALUE_DPmax,prop_not_enough_data_DPmax,...,aff_cat_zero_perc,total_cells_for_aff,smod_cat_10_perc,smod_cat_11_perc,smod_cat_12_perc,smod_cat_13_perc,smod_cat_21_perc,smod_cat_22_perc,smod_cat_23_perc,smod_cat_30_perc
0,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.333333,0.000000,0.666667,0.000000,0.000000,0.000000,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,34.0,0.081081,0.000000,0.000000,0.000000,0.000000,0.918919,0.000000,0.0
3,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.384615,0.000000,0.615385,0.000000,0.000000,0.000000,0.000000,0.0
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,27.0,0.357664,0.343066,0.043796,0.029197,0.029197,0.036496,0.160584,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3793,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.354167,0.576389,0.034722,0.034722,0.000000,0.000000,0.000000,0.0
3794,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.083333,0.916667,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3795,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3796,1.0,0.0,NaN,NaN,1.0,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [160]:
# for clarity, storm name list with number included like: `bopha_0`
for storm in storm_names:
    print(storm)

    if storm == 'nock_ten_20':
        storm_name = 'NOCK-TEN'
    else:
        storm_name = storm.split("_")[0].upper()
    file = f'../data/upscale_data_final_urban/{storm.split("_")[0]}/grided_data_for_{storm}_urban.gpkg'
    gdf_update = gpd.read_file(file)
    gdf_update['typhoon_name'] = storm_name
    gdf_update['typhoon_year'] = events[events['STORMNAME'] == storm_name].start_time.iloc[0][:4]
    gdf_update['grid_point_id'] = gdf_update['id'] #not sure why I didn't do a normal rename function ??????
    aff_list = ['no_aff_data','affected_perc', 'cert_aff', 'prob_aff', 'cert_not_aff',
                'cert_not_aff_perc','pot_affected_perc','cert_aff_perc',
                'aff_cat_four_perc','aff_cat_zero_perc', 'total_cells_for_aff']
    gdf_update[aff_list] = gdf_update.apply(lambda row: affected_perc_and_bool(row['categories_aff']), axis=1, result_type='expand')
    gdf_update.to_file(f'../data/upscale_data_final_urban/{storm.split("_")[0]}/grided_data_for_{storm}_expanded_urban.gpkg', driver='GPKG')
    gdf_update.drop(columns=['geometry','Longitude','Latitude','Centroid','categories_aff','id']).to_csv(f'../data/upscale_data_final_urban/{storm.split("_")[0]}/grided_data_for_{storm}_expanded_urban.csv', index=False)

bopha_0
utor_1
trami_2
usagi_3
nari_4
krosa_5
haiyan_6
lingling_7
sarika_18
haima_19
nock_ten_20
mangkhut_21


Now doing a similar thing, but added the landcover types with zonal stats:

Categories:
- `30` - Urban Centre: Dense urban, high population density
- `23` - Dense Urban Cluster: Semi-dense urban areas
- `22` - Semi-dense Urban Cluster: Moderate density urban
- `21` - Suburban/Peri-urban: Suburban fringes
- `13` - Rural Cluster: Small rural settlements
- `12` - Low Density Rural: Scattered rural population
- `11` - Very Low Density Rural: Very sparse population
- `10` - Water: Water bodies
- `0`  - No Data / Uninhabited: (_Potentially_) No population

In [ ]:
ds = xr.open_dataset(f"../{output_base}/landcover/{region}_smod_all_interp.nc")
ds_interp = ds.interp(year=np.arange(2010, 2021), method="linear")

# lock interpolated values back to nearest valid SMOD category
valid_smod_cats = np.array([10, 11, 12, 13, 21, 22, 23, 30])

def lock_to_nearest_smod(value):
    if np.isnan(value):
        return np.nan
    return valid_smod_cats[np.argmin(np.abs(valid_smod_cats - value))]

snapped = np.vectorize(lock_to_nearest_smod)(ds_interp["smod"].values)

ds_snapped = xr.Dataset(
    {"smod": (["year", "y", "x"], snapped)},
    coords=ds_interp.coords
)

# replaces the unsnapped version
nc_all_path = f"../{output_base}/landcover/{region}_smod_all_interp_2.nc"
ds_snapped.to_netcdf(nc_all_path)
print(f"Saved locked/rounded categorical SMOD NC: {nc_all_path}")

In [145]:
nc_all = xr.open_dataset(f"../{output_base}/landcover/{region}_smod_all_interp_2.nc")

for name in storm_names:
    if name == 'nock_ten_20':
        storm_name = 'NOCK-TEN'
    else:
        storm_name = name.split("_")[0].upper()

    storm_year   = int(events[events['STORMNAME'] == storm_name].start_time.iloc[0][:4])
    clamped_year = max(2010, min(2020, storm_year))
    all_slice    = nc_all["smod"].sel(year=clamped_year).values

    gdf = gpd.read_file(f'../data/upscale_data_final/{name.split("_")[0]}/grided_data_for_{name}_expanded.gpkg')

    all_smod_tif_path = write_temp_tif(all_slice, hazard_tif)
    smod_stats = zonal_stats(gdf, all_smod_tif_path, stats=[], categorical=True, nodata=np.nan)
    os.unlink(all_smod_tif_path)

    all_cats = sorted(set(k for s in smod_stats for k in s.keys() if k is not None))

    for cat in all_cats:
        gdf[f"smod_cat_{int(cat)}_perc"] = [
            (s.get(cat, 0) / sum(s.values())) if sum(s.values()) > 0 else np.nan
            for s in smod_stats
        ]

    gdf["smod_urban_suburban_perc"] = [
        ((s.get(all_cats[-2], 0) + s.get(all_cats[-1], 0) +  s.get(all_cats[-3], 0) + s.get(all_cats[-4], 0)) / sum(s.values())) if sum(s.values()) > 0 else np.nan
        for s in smod_stats
    ] 

    gdf.to_file(
        f'../data/upscale_data_final/{name.split("_")[0]}/grided_data_for_{name}_expanded_landcov.gpkg',
        driver='GPKG'
    )
    gdf.drop(columns=['geometry', 'Longitude', 'Latitude', 'Centroid', 'categories_aff','id']).to_csv(
        f'../data/upscale_data_final/{name.split("_")[0]}/grided_data_for_{name}_expanded_landcov.csv',
        index=False
    )
    print(f"Added SMOD category columns for {name}")

nc_all.close()

Added SMOD category columns for bopha_0
Added SMOD category columns for utor_1
Added SMOD category columns for trami_2
Added SMOD category columns for usagi_3
Added SMOD category columns for nari_4
Added SMOD category columns for krosa_5
Added SMOD category columns for haiyan_6
Added SMOD category columns for lingling_7
Added SMOD category columns for sarika_18
Added SMOD category columns for haima_19
Added SMOD category columns for nock_ten_20
Added SMOD category columns for mangkhut_21


In [161]:
nc_all = xr.open_dataset(f"../{output_base}/landcover/{region}_smod_all_interp_2.nc")

for name in storm_names:
    if name == 'nock_ten_20':
        storm_name = 'NOCK-TEN'
    else:
        storm_name = name.split("_")[0].upper()

    storm_year   = int(events[events['STORMNAME'] == storm_name].start_time.iloc[0][:4])
    clamped_year = max(2010, min(2020, storm_year))
    all_slice    = nc_all["smod"].sel(year=clamped_year).values

    gdf = gpd.read_file(f'../data/upscale_data_final_urban/{name.split("_")[0]}/grided_data_for_{name}_expanded_urban.gpkg')

    all_smod_tif_path = write_temp_tif(all_slice, hazard_tif)
    smod_stats = zonal_stats(gdf, all_smod_tif_path, stats=[], categorical=True, nodata=np.nan)
    os.unlink(all_smod_tif_path)

    all_cats = sorted(set(k for s in smod_stats for k in s.keys() if k is not None))

    for cat in all_cats:
        gdf[f"smod_cat_{int(cat)}_perc"] = [
            (s.get(cat, 0) / sum(s.values())) if sum(s.values()) > 0 else np.nan
            for s in smod_stats
        ]

    gdf.to_file(
        f'../data/upscale_data_final_urban/{name.split("_")[0]}/grided_data_for_{name}_expanded_urban_2.gpkg',
        driver='GPKG'
    )
    gdf.drop(columns=['geometry', 'Longitude', 'Latitude', 'Centroid', 'categories_aff','id']).to_csv(
        f'../data/upscale_data_final_urban/{name.split("_")[0]}/grided_data_for_{name}_expanded_urban_2.csv',
        index=False
    )
    print(f"Added SMOD category columns for {name}")

nc_all.close()

Added SMOD category columns for bopha_0
Added SMOD category columns for utor_1
Added SMOD category columns for trami_2
Added SMOD category columns for usagi_3
Added SMOD category columns for nari_4
Added SMOD category columns for krosa_5
Added SMOD category columns for haiyan_6
Added SMOD category columns for lingling_7
Added SMOD category columns for sarika_18
Added SMOD category columns for haima_19
Added SMOD category columns for nock_ten_20
Added SMOD category columns for mangkhut_21


#### Merging the DataFrames with the target data

The next few cells read in the target data and make sure that they are joined with our now upscaled data that we've created in the last few steps.

**Read in the target data and only keep needed columns:**

In [6]:
# read in the target file and filter to the columns we need for the model training
target_cols = ['typhoon_name', 'typhoon_year', 'grid_point_id','percent_houses_damaged','percent_houses_damaged_5years']
target = pd.read_csv('../data/target/new_model_training_dataset.csv')
# target = target[target_cols]

target.columns

Index(['typhoon_name', 'typhoon_year', 'grid_point_id', 'wind_speed',
       'track_distance', 'rainfall_max_6h', 'rainfall_max_24h', 'total_houses',
       'rwi', 'strong_roof_strong_wall', 'strong_roof_light_wall',
       'strong_roof_salvage_wall', 'light_roof_strong_wall',
       'light_roof_light_wall', 'light_roof_salvage_wall',
       'salvaged_roof_strong_wall', 'salvaged_roof_light_wall',
       'salvaged_roof_salvage_wall', 'mean_slope', 'std_slope', 'mean_tri',
       'std_tri', 'mean_elev', 'coast_length', 'with_coast', 'urban', 'rural',
       'water', 'total_pop', 'percent_houses_damaged',
       'percent_houses_damaged_5years'],
      dtype='str')

**Creating one single dataframe with all of our dataframes we created in pervious cells and the target data:**

In [180]:
dfs = []

# for clarity, storm name list with number included like: `bopha_0`
for storm in storm_names:
    df = pd.read_csv(f'{path}{storm.split("_")[0]}/grided_data_for_{storm}_expanded.csv')
    # df["perc_available_upscaled_new"]
    dfs.append(df)

result= pd.concat(dfs)

final_df = pd.merge(target, result, how="inner", on=["typhoon_name", "typhoon_year", "grid_point_id"])

final_df.to_csv('../data/target/model_training_data_final_urban_only.csv', index=False)

In [ ]:
#with the landcover stuff
dfs = []

# for clarity, storm name list with number included like: `bopha_0`
for storm in storm_names:
    df = pd.read_csv(f'../data/upscale_data_final/{storm.split("_")[0]}/grided_data_for_{storm}_expanded_landcov.csv')
    # df["perc_available_upscaled_new"]
    dfs.append(df)

result= pd.concat(dfs)

final_df = pd.merge(target, result, how="inner", on=["typhoon_name", "typhoon_year", "grid_point_id"])

final_df.to_csv('../data/target/model_training_data_final_with_lc.csv', index=False)

In [162]:
#with the landcover stuff
dfs = []

# for clarity, storm name list with number included like: `bopha_0`
for storm in storm_names:
    df = pd.read_csv(f'../data/upscale_data_final_urban/{storm.split("_")[0]}/grided_data_for_{storm}_expanded_urban_2.csv')
    # df["perc_available_upscaled_new"]
    dfs.append(df)

result= pd.concat(dfs)

final_df = pd.merge(target, result, how="inner", on=["typhoon_name", "typhoon_year", "grid_point_id"])

final_df.to_csv('../data/target/model_training_data_final_urban.csv', index=False)

In [ ]:
storm = 'bopha_0'
df = gpd.read_file(f'{path}{storm.split("_")[0]}/grided_data_for_{storm}_expanded.gpkg')
# df["perc_available_upscaled_new"]

final_df = gpd.merge(target, result, how="inner", on=["typhoon_name", "typhoon_year", "grid_point_id"])

final_df.to_file(
        f'../data/for_vis/grided_data_for_{storm}_expanded_urban_2.gpkg',
        driver='GPKG'
    )

In [10]:
df = pd.read_csv('../data/target/model_training_data_final_urban.csv')

df[df.typhoon_name == 'BOPHA']['pot_affected_perc'].describe()

count    3622.000000
mean        0.027471
std         0.080165
min         0.000000
25%         0.000000
50%         0.000000
75%         0.010296
max         1.000000
Name: pot_affected_perc, dtype: float64

In [14]:
import geopandas as gpd
import os

path = '../data/upscale_data_final/'

os.makedirs('../data/for_vis', exist_ok=True)

for storm in storm_names:
    gpkg_path = f'{path}{storm.split("_")[0]}/grided_data_for_{storm}_expanded_landcov.gpkg'

    if not os.path.exists(gpkg_path):
        print(f"Skipping {storm} — file not found")
        continue

    gdf = gpd.read_file(gpkg_path)

    # keep only what's needed for visualisation
    gdf = gdf[['id', 'pot_affected_perc', 'geometry']]

    out_path = f'../data/for_vis/{storm.split("_")[0]}/grided_data_for_{storm}_pot_aff.gpkg'
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    gdf.to_file(out_path, driver='GPKG')
    print(f"Saved {storm} — {len(gdf)} rows")

Saved bopha_0 — 3798 rows
Saved utor_1 — 3798 rows
Saved trami_2 — 3798 rows
Saved usagi_3 — 3798 rows
Saved nari_4 — 3798 rows
Saved krosa_5 — 3798 rows
Saved haiyan_6 — 3798 rows
Saved lingling_7 — 3798 rows
Saved sarika_18 — 3798 rows
Saved haima_19 — 3798 rows
Saved nock_ten_20 — 3798 rows
Saved mangkhut_21 — 3798 rows


In [167]:
final_df[final_df['no_aff_data'] == 0]

,typhoon_name,typhoon_year,grid_point_id,percent_houses_damaged,percent_houses_damaged_5years,prop_NODATA_VALUE_aff,prop_not_enough_data_aff,majority_aff,minority_aff,prop_NODATA_VALUE_available,...,aff_cat_zero_perc,total_cells_for_aff,smod_cat_10_perc,smod_cat_11_perc,smod_cat_12_perc,smod_cat_13_perc,smod_cat_21_perc,smod_cat_22_perc,smod_cat_23_perc,smod_cat_30_perc
29,BOPHA,2012,5303,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,5.0,0.000000,0.878472,0.090278,0.022569,0.008681,0.000000,0.000000,0.0
30,BOPHA,2012,5304,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,29.0,0.053476,0.467914,0.283422,0.117647,0.053476,0.000000,0.024064,0.0
33,BOPHA,2012,5467,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,30.0,0.222642,0.430189,0.218868,0.015094,0.000000,0.113208,0.000000,0.0
34,BOPHA,2012,5468,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,13.0,0.044484,0.820285,0.085409,0.026690,0.000000,0.023132,0.000000,0.0
35,BOPHA,2012,5469,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,0.800000,15.0,0.000000,0.817708,0.131944,0.024306,0.000000,0.026042,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39835,MANGKHUT,2018,20516,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,7.0,0.052632,0.157895,0.421053,0.000000,0.000000,0.368421,0.000000,0.0
39836,MANGKHUT,2018,20676,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,7.0,0.327869,0.245902,0.278689,0.032787,0.000000,0.114754,0.000000,0.0
39837,MANGKHUT,2018,20677,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,36.0,0.032000,0.176000,0.472000,0.000000,0.128000,0.040000,0.152000,0.0
39840,MANGKHUT,2018,20680,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.000000,15.0,0.086022,0.467742,0.301075,0.064516,0.000000,0.080645,0.000000,0.0


#### Clean and save data for training

The below cells are optional, but they're clearning and processing our data to make sure that it's consitient across the notebooks. However, these functions have also been utilized across notebooks along with the same seed value when it comes to the `train-test split``

In [41]:
# Now, cleaning all the data for easier use in our other modeling notebooks:
df = pd.read_csv('../data/target/model_training_data_new_2.csv')
y_1 = df[['percent_houses_damaged']]
y_2 = df[['percent_houses_damaged_5years']]
X = df.drop(columns=['percent_houses_damaged_5years','percent_houses_damaged'])

X_train_1, X_test_1, y_train_1, y_test_1 = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='simple',
                                                                remove_corr_cols=True,thresh=0.8)

X_train_1.to_csv('../data/cleaned_model_data/simple_filling/x_train_removed_corr_80.csv')
X_test_1.to_csv('../data/cleaned_model_data/simple_filling/x_test_removed_corr_80.csv')
y_train_1.to_csv('../data/cleaned_model_data/simple_filling/y_train_removed_corr_80.csv')
y_test_1.to_csv('../data/cleaned_model_data/simple_filling/y_test_removed_corr_80.csv')

In [23]:
X_train_1, X_test_1, y_train_1, y_test_1 = clean_and_split_data(final_df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='simple')

X_train_1.to_csv('../data/cleaned_model_data/simple_filling/final/x_train.csv')
X_test_1.to_csv('../data/cleaned_model_data/simple_filling/final/x_test.csv')
y_train_1.to_csv('../data/cleaned_model_data/simple_filling/final/y_train.csv')
y_test_1.to_csv('../data/cleaned_model_data/simple_filling/final/y_test.csv')

In [43]:
X_train_1, X_test_1, y_train_1, y_test_1 = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex',
                                                                remove_corr_cols=True,thresh=0.8)

X_train_1.to_csv('../data/cleaned_model_data/complex_filling/x_train_removed_corr_80.csv')
X_test_1.to_csv('../data/cleaned_model_data/complex_filling/x_test_removed_corr_80.csv')
y_train_1.to_csv('../data/cleaned_model_data/complex_filling/y_train_removed_corr_80.csv')
y_test_1.to_csv('../data/cleaned_model_data/complex_filling/y_test_removed_corr_80.csv')

In [24]:
X_train_1, X_test_1, y_train_1, y_test_1 = clean_and_split_data(final_df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex')

X_train_1.to_csv('../data/cleaned_model_data/complex_filling/final/x_train.csv')
X_test_1.to_csv('../data/cleaned_model_data/complex_filling/final/x_test.csv')
y_train_1.to_csv('../data/cleaned_model_data/complex_filling/final/y_train.csv')
y_test_1.to_csv('../data/cleaned_model_data/complex_filling/final/y_test.csv')

In [ ]:
df = pd.read_csv('../data/target/model_training_data_new_3.csv')
X_train_1, X_test_1, y_train_1, y_test_1 = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex')

X_train_1.to_csv('../data/cleaned_model_data/complex_filling/x_train_2.csv')
X_test_1.to_csv('../data/cleaned_model_data/complex_filling/x_test_2.csv')
y_train_1.to_csv('../data/cleaned_model_data/complex_filling/y_train.csv')
y_test_1.to_csv('../data/cleaned_model_data/complex_filling/y_test.csv')

**Calculating zonal stats for the different land covers similar to what was done for affected type above**

In [ ]:
# tried a method to interpolate between the years with landcover (obvisouly assuming a linear change here/oversimplification)
smod_snap_years = {
    'ghsl_smod_2010': 2010,
    'ghsl_smod_2015': 2015,
    'ghsl_smod_2020': 2020,
}

year_arrays = {}
for dim, snap_year in smod_snap_years.items():
    tif_path = f"{output_base}/landcover/{region}_{dim}.tif"
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        data[data == NODATA_VALUE] = np.nan
    year_arrays[snap_year] = data
    print(f"Loaded {dim} for year {snap_year}, unique values: {np.unique(data[~np.isnan(data)])}")

snap_years = sorted(year_arrays.keys())
data_stack = np.stack([year_arrays[y] for y in snap_years], axis=0)

ds = xr.Dataset(
    {"smod": (["year", "y", "x"], data_stack)},
    coords={"year": snap_years,
            "y":    np.arange(data_stack.shape[1]),
            "x":    np.arange(data_stack.shape[2])}
)

ds_interp = ds.interp(year=np.arange(2010, 2021), method="linear")
ds_interp.to_netcdf(f"../data/output_new/landcover/{region}_smod_all_interp.nc")
print("Rebuilt smod_all_interp.nc from full categorical layers")

Running this for visualizations for presentation:

In [ ]:
tiles = ["h30v08", "h30v07", "h30v06", "h29v08", "h29v07", "h29v06"]
year  = 2018
doy   = 305

# write each tile to temp TIF and merge
tmp_files = []
for tile in tiles:
    nc_path = f"../data/tiles/{year}/{tile}_{year}.nc"
    ds      = xr.open_dataset(nc_path)
    da      = ds["ntl"].isel(time=doy - 1)
    da      = da.rio.set_spatial_dims(x_dim="x", y_dim="y").rio.write_crs("EPSG:4326")
    tmp     = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    da.rio.to_raster(tmp.name)
    tmp_files.append(tmp.name)
    print(f"Written tile {tile}")

src_files       = [rasterio.open(f) for f in tmp_files]
mosaic, transform = merge(src_files)

# 
# fig, ax = plt.subplots(figsize=(12, 10))
# img = mosaic[0]
# img_plot = np.where(img <= 0, np.nan, img)  # mask zeros for cleaner viz
# vmax = np.nanpercentile(img_plot, 99)        # clip to 99th percentile so bright spots don't dominate

# ax.imshow(img_plot, cmap="hot", vmin=0, vmax=vmax)
# ax.set_title(f"NTL mosaic — {year} DOY {doy}")
# ax.set_axis_off()
# plt.colorbar(ax.images[0], ax=ax, label="NTL radiance", shrink=0.7)
# plt.tight_layout()
# plt.savefig(f"ntl_mosaic_{year}_doy{doy}.png", dpi=150, bbox_inches="tight")
# plt.close()
# print("Saved visualisation")

# save as a tif
profile = src_files[0].profile.copy()
profile.update({
    "driver":    "GTiff",
    "height":    mosaic.shape[1],
    "width":     mosaic.shape[2],
    "transform": transform,
    "crs":       CRS.from_epsg(4326),
    "dtype":     "float64",
    "compress":  "lzw",
    "count":     1,
})

output_path = f"../data/ntl_mosaic_{year}_doy{doy}.tif"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(mosaic[0], 1)
print(f"Saved TIF: {output_path}")

# cleanup temp files
for src in src_files:
    src.close()
for f in tmp_files:
    os.unlink(f)

Written tile h30v08
Written tile h30v07
Written tile h30v06
Written tile h29v08
Written tile h29v07
Written tile h29v06
Saved TIF: ../data/ntl_mosaic_2018_doy305.tif
